# Web Research Agent Exploration

This notebook explains the web research agent in small, runnable steps. Run the cells from top to bottom. The search and synthesis cells call Tavily and OpenAI, so valid API keys are required.

## 1. Load dependencies and environment

The production script loads `.env` values before creating the Tavily and OpenAI clients.

In [1]:
import os
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages

load_dotenv()

print("Dependencies imported successfully.")

Dependencies imported successfully.


In [ ]:
# Check configuration without displaying secret values.
required_keys = ["OPENAI_API_KEY", "TAVILY_API_KEY"]
configuration_status = {
    key: "configured" if os.getenv(key) else "missing"
    for key in required_keys
}
configuration_status

## 2. Define the shared research state

LangGraph passes this state from the search node to the synthesis node.

In [ ]:
class ResearchState(TypedDict):
    """State shared by the search and synthesis stages."""

    messages: Annotated[list, add_messages]
    query: str
    search_results: list[dict]
    report: str


query = "latest trends in AI and agentic AI"
state = {
    "query": query,
    "messages": [],
    "search_results": [],
    "report": "",
}

state

## 3. Search the web

This cell mirrors `search_web` from `agent.py`. It makes the external Tavily call and stores the raw response for inspection.

In [ ]:
search_tool = TavilySearch(max_results=5)
raw_results = search_tool.invoke(query)

print(f"Raw response type: {type(raw_results).__name__}")
raw_results

In [ ]:
# Normalize Tavily's possible response shapes to a list of dictionaries.
if isinstance(raw_results, dict):
    search_results = raw_results.get("results", [])
elif isinstance(raw_results, list):
    search_results = raw_results
else:
    search_results = []

state["search_results"] = search_results

print(f"Normalized results: {len(search_results)}")
for index, result in enumerate(search_results, start=1):
    print(f"{index}. {result.get('title', 'Untitled')} - {result.get('url', 'N/A')}")

## 4. Build the synthesis prompt

The production agent keeps source excerpts short so the model receives focused evidence.

In [ ]:
results_text = "\n\n".join(
    f"Source: {result.get('url', 'N/A')}\n"
    f"Title: {result.get('title', 'N/A')}\n"
    f"Content: {result.get('content', '')[:500]}"
    for result in search_results
)

messages = [
    SystemMessage(
        content=(
            "You are a research analyst. Synthesize the search results into "
            "a clear, structured report with: Summary, Key Findings "
            "(bullet points), and Sources."
        )
    ),
    HumanMessage(
        content=f"Research query: {query}\n\nSearch results:\n{results_text}"
    ),
]

print(f"Prepared {len(messages)} messages for synthesis.")
print(messages[1].content[:1000])

## 5. Generate a report directly

This optional cell calls the model without LangGraph so the synthesis step can be understood in isolation.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
response = llm.invoke(messages)

report = response.content
print(report)

## 6. Rebuild the LangGraph workflow

This graph is the same sequence used by the CLI and Streamlit application: search, then synthesize, then finish.

In [ ]:
def search_web(state: ResearchState) -> ResearchState:
    """Search Tavily and normalize the response into the graph state."""

    tool = TavilySearch(max_results=5)
    raw_results = tool.invoke(state["query"])

    if isinstance(raw_results, dict):
        results = raw_results.get("results", [])
    elif isinstance(raw_results, list):
        results = raw_results
    else:
        results = []

    return {"search_results": results}


def synthesize_report(state: ResearchState) -> ResearchState:
    """Create a report from the search results using the chat model."""

    results_text = "\n\n".join(
        f"Source: {result.get('url', 'N/A')}\n"
        f"Title: {result.get('title', 'N/A')}\n"
        f"Content: {result.get('content', '')[:500]}"
        for result in state["search_results"]
    )
    messages = [
        SystemMessage(
            content=(
                "You are a research analyst. Synthesize the search results "
                "into a clear, structured report with: Summary, Key Findings "
                "(bullet points), and Sources."
            )
        ),
        HumanMessage(
            content=f"Research query: {state['query']}\n\nSearch results:\n{results_text}"
        ),
    ]
    response = ChatOpenAI(model="gpt-4o-mini", temperature=0).invoke(messages)
    return {"report": response.content, "messages": [response]}

In [ ]:
graph = StateGraph(ResearchState)
graph.add_node("search", search_web)
graph.add_node("synthesize", synthesize_report)
graph.set_entry_point("search")
graph.add_edge("search", "synthesize")
graph.add_edge("synthesize", END)

agent = graph.compile()
print("Graph compiled successfully.")

## 7. Run the complete agent

This final cell invokes the compiled graph with the same initial state used by `agent.py` and returns the report plus search results.

In [ ]:
final_state = agent.invoke(
    {
        "query": query,
        "messages": [],
        "search_results": [],
        "report": "",
    }
)

print(final_state["report"])
print(f"\nSources returned: {len(final_state['search_results'])}")